# MIMIC-IV Demo: Comprehensive Clinical Deep Dive & Temporal Analysis

This notebook provides a detailed Exploratory Data Analysis (EDA) of the MIMIC-IV Demo dataset. Unlike basic EDAs, this analysis focuses on:
- **Clinical Trajectories:** Patient flow from Emergency Department (ED) to Intensive Care (ICU).
- **Adverse Event Signals:** Linking medications to lab value trajectories (e.g., renal/liver function).
- **Fairness Audit:** Assessing biases in insurance, language, and boarding times.
- **OMOP Readiness:** Mapping MIMIC codes to standard terminologies (SNOMED, RxNorm, LOINC).

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import gzip
import glob
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('bmh')

# Paths setup
mimic_path = r'D:\ADE DATASET DOWNLOAD\MIMIC-IV-Demo_extracted'
vocab_path = r'D:\ADE DATASET DOWNLOAD\omop vocab'

def load_mimic_table(module, table_name):
    """
    Robustly loads a MIMIC table by searching through typical module structures.
    """
    module_search_path = os.path.join(mimic_path, module)
    
    # Standard patterns for compressed and uncompressed files
    search_patterns = [
        os.path.join(module_search_path, "**", f"{table_name}.csv.gz"),
        os.path.join(module_search_path, "**", f"{table_name}.csv")
    ]
    
    found_files = []
    for pattern in search_patterns:
        found_files.extend(glob.glob(pattern, recursive=True))
    
    if not found_files:
        # Fallback: search the entire mimic_path if not found in specific module
        found_files.extend(glob.glob(os.path.join(mimic_path, "**", f"{table_name}.csv*"), recursive=True))
        
    if not found_files:
        print(f"[!] File not found for: {table_name}")
        return pd.DataFrame()
    
    path = found_files[0]
    compression = 'gzip' if path.endswith('.gz') else None
    return pd.read_csv(path, compression=compression, low_memory=False)

## 1. Core Module Analysis
Analyzing patient demographics and admissions structure.

In [ ]:
patients = load_mimic_table('clinical', 'patients')
admissions = load_mimic_table('clinical', 'admissions')
transfers = load_mimic_table('clinical', 'transfers')

print(f"Patients: {len(patients)} | Admissions: {len(admissions)} | Transfers: {len(transfers)}")

if not patients.empty:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    sns.histplot(patients['anchor_age'], bins=30, kde=True, ax=ax[0], color='teal')
    ax[0].set_title('Patient Age Distribution')
    
    patients['gender'].value_counts().plot.pie(autopct='%1.1f%%', ax=ax[1], colors=['#66b3ff','#99ff99'])
    ax[1].set_title('Gender Balance')
    display(patients.head(3))

if not admissions.empty:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=admissions, y='admission_type', palette='viridis')
    plt.title('Admission Types')
    display(admissions.head(3))

## 2. Hospital Module (hosp): Deep Dive
Focusing on Diagnoses (ICD), Procedures, and Prescriptions.

In [ ]:
diagnoses = load_mimic_table('clinical', 'diagnoses_icd')
procedures = load_mimic_table('clinical', 'procedures_icd')
prescriptions = load_mimic_table('clinical', 'prescriptions')

if not diagnoses.empty:
    plt.figure(figsize=(12, 5))
    top_dx = diagnoses['icd_code'].value_counts().head(15)
    sns.barplot(x=top_dx.values, y=top_dx.index, palette='magma')
    plt.title('Top 15 Diagnosis Codes (ICD)')

if not prescriptions.empty:
    # Medication dose/route analysis
    # Note: 'frequency' is missing in some demo versions, using 'doses_per_24_hrs'
    presc_cols = ['drug', 'dose_val_rx', 'dose_unit_rx', 'route', 'doses_per_24_hrs']
    available_cols = [c for c in presc_cols if c in prescriptions.columns]
    display(prescriptions[available_cols].head(5))
    
    plt.figure(figsize=(10, 4))
    if 'route' in prescriptions.columns:
        prescriptions['route'].value_counts().head(10).plot(kind='barh', color='coral')
        plt.title('Common Medication Routes')

## 3. ICU Module: High-Acuity Data
Analyzing chart events, vital signs, and severity proxies (Ventilator/Vasopressor).

In [ ]:
chartevents = load_mimic_table('icu', 'chartevents')
d_items = load_mimic_table('icu', 'd_items')

if not chartevents.empty and not d_items.empty:
    # Join with metadata to see what we are looking at
    chart_labeled = chartevents.merge(d_items[['itemid', 'label', 'abbreviation']], on='itemid')
    print(f"Total ICU Chart Events: {len(chart_labeled)}")
    
    top_icu_labels = chart_labeled['label'].value_counts().head(10)
    plt.figure(figsize=(12, 5))
    sns.barplot(x=top_icu_labels.values, y=top_icu_labels.index, palette='flare')
    plt.title('Most Frequent ICU Charted Parameters')
    
    # Sample vital sign trajectory (e.g., Heart Rate)
    hr_data = chart_labeled[chart_labeled['label'].str.contains('Heart Rate', na=False)]
    if not hr_data.empty:
        sample_subject = hr_data['subject_id'].iloc[0]
        subj_hr = hr_data[hr_data['subject_id'] == sample_subject].sort_values('charttime')
        plt.figure(figsize=(15, 4))
        plt.plot(pd.to_datetime(subj_hr['charttime']), subj_hr['valuenum'], marker='o', linestyle='-', color='red')
        plt.title(f'Heart Rate Trajectory: Patient {sample_subject}')
        plt.xlabel('Time')
        plt.ylabel('BPM')
        plt.xticks(rotation=45)

## 4. ED Module: Emergency Admissions
Boarding times and Triage acuity.

In [ ]:
edstays = load_mimic_table('ed', 'edstays')
triage = load_mimic_table('ed', 'triage')

if not edstays.empty:
    edstays['intime'] = pd.to_datetime(edstays['intime'])
    edstays['outtime'] = pd.to_datetime(edstays['outtime'])
    edstays['boarding_hours'] = (edstays['outtime'] - edstays['intime']).dt.total_seconds() / 3600
    
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    sns.histplot(edstays['boarding_hours'].dropna(), bins=30, kde=True, ax=ax[0], color='orange')
    ax[0].set_title('ED Boarding Time Distribution (Hours)')
    
    if not triage.empty:
        sns.countplot(data=triage, x='acuity', palette='Reds', ax=ax[1])
        ax[1].set_title('Triage Acuity (1=Highest, 5=Lowest)')

## 5. Temporal Analysis: Medication vs. Labs
Searching for Adverse Drug Event (ADE) signals by linking medication timing to renal function (Creatinine).

In [ ]:
labevents = load_mimic_table('clinical', 'labevents')
d_labitems = load_mimic_table('clinical', 'd_labitems')

if not labevents.empty and not prescriptions.empty:
    # Filter for Creatinine (Renal Function)
    creat_id = d_labitems[d_labitems['label'].str.contains('Creatinine', case=False, na=False)]['itemid'].unique()
    labs_creat = labevents[labevents['itemid'].isin(creat_id)]
    
    # Select a patient with both prescriptions and creatinine labs
    common_patients = set(prescriptions['subject_id']).intersection(set(labs_creat['subject_id']))
    if common_patients:
        subj = list(common_patients)[0]
        subj_labs = labs_creat[labs_creat['subject_id'] == subj].sort_values('charttime')
        subj_meds = prescriptions[prescriptions['subject_id'] == subj].sort_values('starttime')
        
        plt.figure(figsize=(15, 6))
        plt.plot(pd.to_datetime(subj_labs['charttime']), subj_labs['valuenum'], marker='s', label='Creatinine (mg/dL)', color='brown')
        
        # Add vertical lines for medication starts
        for i, row in subj_meds.head(10).iterrows():
            plt.axvline(pd.to_datetime(row['starttime']), color='green', alpha=0.3, linestyle='--')
            plt.text(pd.to_datetime(row['starttime']), plt.ylim()[1]*0.9, row['drug'], rotation=90, fontsize=8)
            
        plt.title(f'Temporal Signal: Medication Timing vs. Renal Function (Subject {subj})')
        plt.legend()
        plt.show()

## 6. Fairness & Bias Audit
Analyzing social determinants of health (SDOH) proxies.

In [ ]:
if not admissions.empty:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    
    # Insurance as Socioeconomic Proxy
    admissions['insurance'].value_counts().plot.bar(ax=ax[0], color='skyblue')
    ax[0].set_title('Insurance Distribution')
    
    # Language barriers
    admissions['language'].value_counts().plot.bar(ax=ax[1], color='salmon')
    ax[1].set_title('Primary Language Distribution')
    
    # ED Boarding Time by Insurance (Bias Check)
    if 'boarding_hours' in edstays.columns:
        merged_ed = edstays.merge(admissions[['hadm_id', 'insurance']], on='hadm_id', how='left')
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=merged_ed, x='insurance', y='boarding_hours', palette='Set3')
        plt.title('ED Boarding Time by Insurance Type')
        plt.ylabel('Hours')

## 7. OMOP Mapping Readiness
MIMIC to OMOP Common Data Model (CDM) mapping exploration.

In [ ]:
def load_vocab_table(table_name):
    path = os.path.join(vocab_path, f"{table_name}.csv")
    if os.path.exists(path):
        # OMOP vocab files are typically TSV even if named .csv
        return pd.read_csv(path, sep='\t', low_memory=False)
    return pd.DataFrame()

concept = load_vocab_table('CONCEPT')

if not concept.empty:
    print("OMOP Vocabulary Loaded.")
    # Example: Search for an ICD-10 code (e.g., I10 for Hypertension)
    hypertension_map = concept[concept['concept_code'] == 'I10']
    display(hypertension_map[['concept_id', 'concept_name', 'domain_id', 'vocabulary_id', 'standard_concept']])
    
    # Demonstrate NDC to RxNorm link if data permits
    rxnorm_sample = concept[concept['vocabulary_id'] == 'RxNorm'].head(5)
    print("\nRxNorm Standard Concepts Sample:")
    display(rxnorm_sample[['concept_id', 'concept_name', 'concept_class_id']])

## 8. Summary of Limitations
1. **Sample Size:** This is the Demo dataset (approx. 100 patients) and results are not generalizable.
2. **Missing Notes:** Free-text clinical notes are missing from the demo, limiting NLP potential.
3. **Time Shifting:** MIMIC dates are shifted into the future to protect privacy; intervals are preserved but absolute dates are artificial.
4. **Transportability:** Data is from a single center (BIDMC), which may have unique coding practices.

In [ ]:
print("MIMIC-IV Demo EDA Completed Successfully.")